# Lab 3 — Choosing a pattern from Table 19.3: measure before you move down (EP06)

§19.4 says to use Table 19.3 as a decision matrix: **start from the top row (simplest) and move down only when the simpler pattern demonstrably fails**,
looking at three things: how predictable the task's structure is · how many LLM calls you can afford (time + money) · whether quality needs iteration

**What you'll learn**
1. A meter: count LLM calls, tool calls, tokens, cost and time for each pattern
2. A top-row pattern: **prompt chaining** — fixed steps, 2 LLM calls
3. A middle-row pattern: **ReAct** — the model decides which tools to call, and how many times
4. Task A: is a chain enough? · Task B: does the chain really fail? And is moving down to ReAct worth it?

The paper library (titles and years are real), the summaries and the **learner ratings are simulated data written for teaching**.
Every tool call, model answer and measured number comes from a real run.

> Source: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.4 and Table 19.3 — CC BY-SA 4.0

## 0. Setup

| You need | Details |
|---|---|
| Python 3.10 or later | or Google Colab, which already has it |
| An OpenRouter API key | sign up at openrouter.ai → Keys → Create Key, then add credit (this lab costs well under one cent) |
| Packages | `openai`, `python-dotenv` (installed in the next cell) |

**Adding the API key**
- **Colab:** the 🔑 Secrets icon → name `OPENROUTER_API_KEY` → turn on *Notebook access*
- **Your own machine:** create a `.env` file next to the notebook with the line `OPENROUTER_API_KEY=sk-or-...`

How to set up the key, and why we chose this model, is in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

## 1. The paper library + two tools

The same 12 papers, now with a **summary** and a **learner rating** (out of 5), visible only through a new tool, `read_paper`
`search_papers` returns just the id, title and year (`STOP` already has "learning" — from Lab 2)

In [ ]:
import json, os, time, datetime
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"reasoning": {"enabled": False},
         "provider": {"order": ["deepinfra/fp8"], "allow_fallbacks": False}}

In [ ]:
PAPERS = [
    {"id": "P01", "title": "Informer: efficient transformer for long sequence time-series forecasting", "year": 2021,
     "summary": "ProbSparse self-attention makes transformers fast enough for very long input sequences.", "rating": 4.0},
    {"id": "P02", "title": "Autoformer: decomposition transformers for long-term series forecasting", "year": 2021,
     "summary": "Splits a series into trend and seasonal parts and replaces attention with auto-correlation.", "rating": 3.8},
    {"id": "P03", "title": "FEDformer: frequency enhanced decomposed transformer for time series", "year": 2022,
     "summary": "Combines trend-seasonal decomposition with attention computed in the frequency domain.", "rating": 3.6},
    {"id": "P04", "title": "Are transformers effective for time series forecasting?", "year": 2023,
     "summary": "Shows that a one-layer linear model often beats transformer forecasters on standard benchmarks.", "rating": 4.7},
    {"id": "P05", "title": "PatchTST: a time series is worth 64 words", "year": 2023,
     "summary": "Cuts each series into patches used as tokens, and models each channel independently.", "rating": 4.8},
    {"id": "P06", "title": "iTransformer: inverted transformers for time series forecasting", "year": 2024,
     "summary": "Treats each variable's whole series as one token, so attention runs across variables.", "rating": 4.3},
    {"id": "P07", "title": "TimesFM: a decoder-only foundation model for time-series forecasting", "year": 2024,
     "summary": "A decoder-only model pretrained on a large time-series corpus that forecasts new series zero-shot.", "rating": 4.4},
    {"id": "P08", "title": "Chronos: learning the language of time series", "year": 2024,
     "summary": "Scales and quantizes values into tokens and trains language-model architectures on them.", "rating": 3.9},
    {"id": "P09", "title": "Attention is all you need", "year": 2017,
     "summary": "Introduces the Transformer, built only from attention, for machine translation.", "rating": 4.9},
    {"id": "P10", "title": "BERT: pre-training of deep bidirectional transformers", "year": 2019,
     "summary": "Pretrains a bidirectional transformer encoder on text, then fine-tunes it for NLP tasks.", "rating": 4.5},
    {"id": "P11", "title": "Temporal fusion transformers for interpretable time series forecasting", "year": 2019,
     "summary": "An attention model for multi-horizon forecasting whose weights can be interpreted.", "rating": 3.7},
    {"id": "P12", "title": "Moirai: unified training of universal time series forecasting transformers", "year": 2024,
     "summary": "A universal forecaster pretrained on a large multi-domain dataset, used zero-shot.", "rating": 4.1},
]
STOP = {"with", "from", "paper", "papers", "model", "models", "using", "published", "since", "research", "about", "learning"}

def find_papers(query, year_from=0):
    words = [w for w in query.lower().replace("-", " ").split() if len(w) > 3 and w not in STOP]
    return [{"id": p["id"], "title": p["title"], "year": p["year"]} for p in PAPERS
            if p["year"] >= year_from and any(w in p["title"].lower().replace("-", " ") for w in words)]

def read_paper(paper_id):
    return next((p for p in PAPERS if p["id"] == paper_id), "no paper with this id")

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "search_papers",
        "description": "Search academic papers in the Cogito Learn library by topic. Returns a list of {id, title, year}; returns [] if none match.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "Topic in English, e.g. 'time series transformer'"},
            "year_from": {"type": "integer", "description": "Earliest publication year wanted (CE), e.g. 2023 = from 2023 onward"}},
        "required": ["query", "year_from"]}}},
    {"type": "function", "function": {
        "name": "read_paper",
        "description": "Read one paper's details. Returns {id, title, year, summary, rating}; rating is the average score from Cogito Learn learners (out of 5).",
        "parameters": {"type": "object", "properties": {
            "paper_id": {"type": "string", "description": "An id from search_papers, e.g. 'P05'"}},
        "required": ["paper_id"]}}},
]

def run_tool(name, args):
    if name == "search_papers":
        return find_papers(args["query"], args["year_from"])
    return read_paper(args["paper_id"])

SYSTEM = "You are the Cogito Learn paper-search assistant. Always use tools to get real data; never answer from memory. Today is " + str(datetime.date.today())

## 2. A meter

Every LLM call goes through `llm`, which counts calls and tokens · `measure` runs a pattern once and returns all the numbers

In [ ]:
STATS = {}

def llm(**kwargs):
    reply = client.chat.completions.create(model=MODEL, extra_body=EXTRA, **kwargs)
    STATS["calls"] += 1
    STATS["tokens_in"] += reply.usage.prompt_tokens
    STATS["tokens_out"] += reply.usage.completion_tokens
    return reply.choices[0].message

def measure(pattern, question, check):
    STATS.update(calls=0, tools=0, tokens_in=0, tokens_out=0)
    t0 = time.time()
    answer = pattern(question)
    sec = time.time() - t0
    cost = STATS["tokens_in"] / 1e6 * 0.07 + STATS["tokens_out"] / 1e6 * 0.34
    return check(answer), dict(STATS), cost, sec, answer

def report(pattern, question, check, rounds=3):
    for i in range(rounds):
        ok, s, cost, sec, answer = measure(pattern, question, check)
        print(f"{pattern.__name__:6} {'right' if ok else 'wrong'} · {s['calls']} LLM calls · {s['tools']} tool calls · {s['tokens_in'] + s['tokens_out']:,} tokens · ${cost:.5f} · {sec:.1f} s")
    return answer

## 3. Top row: prompt chaining

Fixed steps, written in code: ① the LLM turns the request into a search query (structured output) → ② search, then read the first result (code does this, not the model) → ③ the LLM answers from that information

In [ ]:
QUERY = {"type": "json_schema", "json_schema": {"name": "query", "strict": True, "schema": {
    "type": "object", "properties": {"query": {"type": "string"}, "year_from": {"type": "integer"}},
    "required": ["query", "year_from"], "additionalProperties": False}}}

def chain(question):
    q = json.loads(llm(response_format=QUERY, messages=[
        {"role": "system", "content": "Turn the request into an English search query and the earliest year wanted (CE; 0 if not given)."},
        {"role": "user", "content": question}]).content)
    hits = find_papers(q["query"], q["year_from"])
    detail = read_paper(hits[0]["id"]) if hits else None
    STATS["tools"] += 2
    prompt = f"Request: {question}\nSearch results: {hits}\nDetails of the first result: {detail}\nAnswer the request from this information only."
    return llm(messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]).content

## 4. Middle row: ReAct

The same agent loop — the model chooses what to search, which papers to read and how many times, until it's ready to answer

In [ ]:
def react(question, max_steps=8):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for step in range(max_steps):
        msg = llm(messages=messages, tools=TOOLS)
        messages.append(msg.model_dump(exclude_none=True))
        if not msg.tool_calls:
            return msg.content
        for call in msg.tool_calls:
            result = run_tool(call.function.name, json.loads(call.function.arguments))
            STATS["tools"] += 1
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": json.dumps(result, ensure_ascii=False)})
    return "stopped: step limit reached"

## 5. Task A: summarize PatchTST in three points

The steps are known in advance (search → read → summarize) · passes when the answer mentions patches

In [ ]:
TASK_A = "Summarize the PatchTST paper in three points"
check_a = lambda answer: "patch" in answer.lower()

answer = report(chain, TASK_A, check_a)
print(answer)

In [ ]:
answer = report(react, TASK_A, check_a)

## 6. Task B: papers from 2023 on that learners rated ≥ 4.5

The right answer: **P04** (4.7) and **P05 PatchTST** (4.8) — but ratings live only in `read_paper`, and we can't know in advance how many papers to read

In [ ]:
TASK_B = "Recommend time-series papers from 2023 onward that learners rated 4.5 or higher"
check_b = lambda answer: "PatchTST" in answer and "effective" in answer

answer = report(chain, TASK_B, check_b)
print(answer)

In [ ]:
answer = report(react, TASK_B, check_b)
print(answer)

## Summary

| Task | chain (top row) | ReAct (middle row) | Choose |
|---|---|---|---|
| A: summarize PatchTST | right 3/3 · 2 LLM calls | right 3/3 · 3 LLM calls · ~3–4× the cost, nearly 2× the time | **chain** |
| B: rating ≥ 4.5 | wrong 3/3 · reads only the first result, misses PatchTST | right 3/3 · 1 search + 6 reads · 3–8 LLM calls (not fixed) · several times the cost | **ReAct** |

Always start at the top row · move down only when it demonstrably fails (measured with a check function) · the higher cost is worth it only when the cheaper pattern is wrong

**Exercises**
1. Change `chain`'s fixed step to read **every** result found (not just the first), then run `report` on task B — does it pass, and with how many LLM calls? If it passes, which pattern should task B use?
2. Try task C: "which paper in the library has the highest rating?" with both patterns, and decide from the numbers

**Next — Lab 4:** all of this EP's design principles again, with LangChain middleware